# 06 容量规划与 Kubernetes 部署基础

本章把“模型能跑”升级成“服务能承载用户”。容量规划不是猜几张卡，而是从 workload、token、延迟、显存、吞吐和安全余量推导部署规模。


## 1. Docker 与 Kubernetes 最小基础

- **Container**：把应用和依赖打包，保证环境一致。
- **Pod**：Kubernetes 最小调度单位，一个 Pod 可包含一个或多个容器。
- **Deployment**：管理 Pod 副本、滚动更新和回滚。
- **Service**：给一组 Pod 提供稳定访问入口。
- **Ingress/Gateway**：把外部 HTTP 流量引入集群。
- **HPA/KEDA**：根据指标扩缩容。
- **GPU resource**：通过 `nvidia.com/gpu` 等资源请求让 Pod 调度到 GPU 节点。

```mermaid
flowchart TB
    User["用户 / User"] --> Ingress["入口网关 / Ingress / Gateway"]
    Ingress --> Service["服务 / Service"]
    Service --> Pod1["Pod 副本 1 / Pod replica 1<br/>vLLM 容器 / container"]
    Service --> Pod2["Pod 副本 2 / Pod replica 2<br/>vLLM 容器 / container"]
    Pod1 --> GPUNode1["GPU 节点 / GPU Node"]
    Pod2 --> GPUNode2["GPU 节点 / GPU Node"]
```


## 2. 容量规划公式

基本关系：

$$
\text{concurrency} \approx \text{QPS} \times \text{average latency seconds}
$$

如果单副本可稳定承载 `C` 个并发，目标利用率为 `u`，副本数粗估：

$$
\text{replicas} = \left\lceil \frac{\text{target concurrency}}{C \times u} \right\rceil
$$

显存预算：

$$
\text{VRAM} \approx \text{weights} + \text{KV cache} + \text{activations} + \text{workspace} + \text{safety margin}
$$

这些公式不是最终答案，而是 benchmark 前的草稿。最终必须用真实 token 分布和压测校准。


In [ ]:
import math

def estimate_replicas(qps, avg_latency_s, capacity_per_replica, target_utilization=0.65):
    concurrency = qps * avg_latency_s
    replicas = math.ceil(concurrency / (capacity_per_replica * target_utilization))
    return concurrency, replicas

scenarios = [
    ('内部助手', 1.5, 5.0, 8),
    ('客服聊天', 10.0, 6.0, 16),
    ('活动峰值', 40.0, 5.0, 24),
]
for name, qps, latency, cap in scenarios:
    conc, reps = estimate_replicas(qps, latency, cap)
    print(f'{name:8s} concurrency≈{conc:6.1f}, replicas≈{reps}')


## 3. GPU Operator 的作用

在 Kubernetes 中，普通 Pod 不会天然知道 GPU。NVIDIA GPU Operator 负责在 GPU 节点上管理驱动、container runtime、device plugin、监控组件等，让集群能把 GPU 作为可调度资源暴露出来。

```mermaid
flowchart LR
    GPUNode["GPU 节点 / GPU node"] --> Driver["NVIDIA 驱动 / NVIDIA driver"]
    GPUNode --> Runtime["容器运行时 / container runtime"]
    GPUNode --> Plugin["设备插件 / device plugin<br/>nvidia.com/gpu"]
    Plugin --> Scheduler["K8s 调度器 / K8s scheduler"]
    Scheduler --> Pod["GPU Pod / GPU Pod<br/>GPU 工作负载"]
```

产品/研发不一定要安装 Operator，但要知道：如果 `nvidia.com/gpu` 不可见、Pod 调度不到 GPU、驱动版本不一致、节点标签缺失，模型服务就无法正常部署。


## 4. KServe 与 InferenceService

KServe 用 `InferenceService` 把模型服务部署、流量、autoscaling、runtime 配置抽象成 Kubernetes CRD。它支持不同 deployment mode、resource defaults、autoscaler class、storage 配置等。

概念化 YAML：

```yaml
apiVersion: serving.kserve.io/v1beta1
kind: InferenceService
metadata:
  name: qwen-vllm
spec:
  predictor:
    containers:
      - name: kserve-container
        image: vllm/vllm-openai:latest
        args:
          - --model
          - Qwen/Qwen2.5-7B-Instruct
          - --served-model-name
          - qwen
        resources:
          limits:
            nvidia.com/gpu: "1"
```

不要只复制 YAML。你要理解每个字段背后的问题：镜像怎么构建、模型怎么下载、GPU 怎么申请、健康检查怎么做、指标怎么采集、扩缩容按什么指标触发。


## 5. 容量规划 walkthrough

假设一个企业 RAG 服务：

- 峰值 QPS：8
- 平均总延迟目标：6 秒
- p95 TTFT 目标：2 秒内
- 平均 prompt：3,000 tokens，p95 prompt：8,000 tokens
- 平均 output：300 tokens
- 单实例稳定承载并发：12
- 目标利用率：65%

并发粗估：`8 * 6 = 48`。副本数：`ceil(48 / (12 * 0.65)) = 7`。这只是起点。接下来要用真实 workload benchmark 检查 p95/p99、OOM、preemption、GPU 利用率和成本。如果 p95 prompt 很长导致 TTFT 高，可能先优化 RAG 压缩，而不是盲目加 GPU。


## 6. 常见误区

- **用平均 token 长度做容量规划。** p95/p99 长 prompt 才是风险来源。
- **只按 QPS 扩容。** LLM 请求差异巨大，要看 token rate 和并发。
- **HPA 指标选错。** CPU 利用率不一定能代表 GPU 推理压力。
- **没有冷启动预算。** 大模型加载可能很慢，scale from zero 对交互式服务未必合适。
- **把单机 benchmark 直接乘副本数。** 网关、队列、下载、存储、网络都可能成为新瓶颈。


## 7. 为什么 LLM autoscaling 比普通 Web 服务难

普通 Web 服务常按 CPU/QPS 扩缩容，但 LLM 服务的压力更接近 token workload：

- 一个短请求和一个长 RAG 请求 QPS 都是 1，但成本可能差十倍以上。
- decode 是持续流式过程，请求占用资源时间长。
- 模型冷启动慢，scale from zero 对聊天体验可能不可接受。
- GPU 是离散资源，不能像 CPU 一样细粒度切分。
- KV cache 会让同样 QPS 在不同上下文长度下显存压力完全不同。

因此更实用的 autoscaling 指标可能包括：queue time、running/waiting requests、token rate、GPU memory/KV cache、TTFT p95，而不是只看 CPU。


In [ ]:
# token-aware capacity：按 token rate 粗估副本，而不是只按 QPS。
def replicas_by_token_rate(qps, prompt_tokens, output_tokens, capacity_tokens_per_s, target_util=0.65):
    token_rate = qps * (prompt_tokens + output_tokens)
    replicas = math.ceil(token_rate / (capacity_tokens_per_s * target_util))
    return token_rate, replicas

cases = [
    ('short chat', 10, 300, 120),
    ('long rag', 10, 6000, 250),
    ('agent long output', 10, 1500, 1200),
]
for name, qps, p, o in cases:
    rate, reps = replicas_by_token_rate(qps, p, o, capacity_tokens_per_s=12000)
    print(f'{name:18s} token_rate={rate:7.0f}/s replicas≈{reps}')


## 8. 上线容量评审模板

一次 LLM 服务上线评审至少要填这些项：

- 模型：名称、参数量、dtype/量化、max context、是否支持工具/结构化输出。
- Workload：QPS、并发、prompt/output token p50/p95/p99、峰值系数。
- SLA：TTFT p95、ITL p95、总延迟 p95、错误率、可用性。
- 资源：单实例 GPU 类型、显存、最大 batch token、并发限制、replica 数。
- 风险：OOM、冷启动、fallback、成本上限、长请求限流、模型下载失败。
- 观测：metrics、logs、traces、request id、dashboard、告警。
- 回滚：旧模型/旧 prompt/旧数据是否可切回。

这个模板能让产品、研发、平台和 SRE 在同一张表上讨论。


## 9. GPU Pod 为什么经常“调度不上”

在 Kubernetes 里，GPU Pod 调度失败常见原因包括：节点没有 GPU label、device plugin 没工作、GPU 已被其他 Pod 占用、资源 request 写错、nodeSelector/toleration 不匹配、驱动或 runtime 配置有问题。排查时先看 Pod event，再看节点 allocatable 资源，再看 GPU Operator 和 device plugin 状态。

对产品/研发来说，理解这些不是为了亲自修集群，而是为了知道“模型服务没起来”可能不是代码 bug，而是资源和平台问题。你在上线计划中要预留模型下载、镜像拉取、GPU 调度、冷启动和回滚时间。


## 10. 压测设计

容量规划最终要靠压测验证。一个好的 LLM 压测至少要包含：

- 真实 prompt 长度分布，而不是固定短 prompt。
- 真实 output 长度或 max_tokens 设置。
- 并发阶梯：逐步提高并发，观察拐点。
- 指标：TTFT p50/p95/p99、ITL p50/p95/p99、throughput、error、OOM、queue time。
- 场景分组：短问答、长 RAG、Agent 工具说明、结构化输出。
- 预热和冷启动分开统计。

压测结论不要只写“QPS 达到多少”，要写“在什么 token 分布和 SLA 下达到多少”。


## 11. 自测题

1. 为什么 LLM autoscaling 不适合只看 CPU？
2. `nvidia.com/gpu: 1` 在 Kubernetes 中表示什么？
3. 为什么 scale from zero 对大模型聊天服务可能有风险？
4. 容量规划为什么要看 p95/p99 token 长度？
5. 一份压测报告至少应该包含哪些指标？


## 12. 从单机到集群的心智转换

单机部署时，你关心的是“这个模型能不能在这张卡上跑”。集群部署时，你关心的是“这个服务在多个副本、多节点、滚动升级、失败恢复、流量波动下是否稳定”。这会引入新问题：模型镜像多大，拉取要多久；模型权重从对象存储下载要多久；Pod 重启时是否影响在线用户；多个副本是否共享 cache；负载均衡是否 session-aware；节点维护时是否有足够冗余。

Kubernetes 提供 Deployment、Service、HPA、资源调度等基础能力，但 LLM 服务还需要额外考虑 GPU 特性。GPU 是昂贵且离散的资源，Pod 申请 1 张 GPU 往往意味着独占整张；模型冷启动可能以分钟计；autoscaling 不能只看 CPU；长连接 streaming 会影响负载均衡。KServe、Ray Serve 和各种 LLM platform 都是在这些基础问题上做更高层抽象。

因此你写容量方案时，要区分三层：模型实例能承载多少，Kubernetes 能否稳定调度这些实例，网关能否把流量合理送到这些实例。任何一层缺失，线上都可能出问题。


## 来源与延伸阅读

本章正文已经把关键内容消化进教程，下面链接只用于延伸阅读和核对最新细节：vLLM、vLLM-Metal、Ray Serve LLM、SGLang、TensorRT-LLM、Text Generation Inference、LiteLLM、KServe、NVIDIA GPU Operator、OpenTelemetry GenAI。
